In [9]:
import os
import joblib
import pandas as pd

from xgboost import XGBClassifier, XGBRegressor


1. Save Case 1 — XGBoost Classifier

In [2]:
# Create folder
os.makedirs("saved_models", exist_ok=True)

print("==========================================")
print("MODEL SAVING")
print("==========================================")

MODEL SAVING


# CASE 1 - CLASSIFICATION

In [11]:
X_train_c1 = pd.read_csv("processed_data/X_train.csv")
y_train_c1=pd.read_csv("processed_data/y_train.csv").values.ravel()


In [13]:
y_train_c1

array([0, 0, 0, ..., 0, 0, 0], shape=(750879,))

In [14]:
# Class imbalance
scale_pos_weight = ((y_train_c1 == 0).sum()/(y_train_c1 == 1).sum())

clf = XGBClassifier(scale_pos_weight=scale_pos_weight,random_state=42,eval_metric="logloss",n_jobs=-1)

clf.fit(X_train_c1, y_train_c1)

# Save model
joblib.dump(clf,"saved_models/best_case1_pha_classifier.joblib")

print("[SAVED] Case 1 model")

[SAVED] Case 1 model


2. Save Case 2 — XGBoost Regressor

 CASE 2 - REGRESSION

In [15]:
X_train_c2 = pd.read_csv("processed_regression/X_train.csv")

y_train_c2 = pd.read_csv("processed_regression/y_train.csv").values.ravel()

reg = XGBRegressor(n_estimators=100,learning_rate=0.1,random_state=42,n_jobs=-1)

reg.fit(X_train_c2, y_train_c2)

# Save model
joblib.dump(reg,"saved_models/best_case2_diameter_regressor.joblib")

print("[SAVED] Case 2 model")

[SAVED] Case 2 model


3. Case 3 — Clustering Experiment Comparison

 CASE 3 - CLUSTERING COMPARISON TABLE

In [5]:
# Create results folder
os.makedirs("results", exist_ok=True)


# K-Means results
# ------------------------------------------------

k_range = range(2, 9)

# Put the silhouette scores you got from your K-Means notebook here
kmeans_silhouette_scores = [0.36,0.37,0.32,0.27,0.29,0.30,0.28]
   

kmeans_results = pd.DataFrame({
    "Model": "K-Means",
    "Parameter_Name": "k",
    "Parameter": list(k_range),
    "Silhouette": kmeans_silhouette_scores})

print(kmeans_results)

     Model Parameter_Name  Parameter  Silhouette
0  K-Means              k          2        0.36
1  K-Means              k          3        0.37
2  K-Means              k          4        0.32
3  K-Means              k          5        0.27
4  K-Means              k          6        0.29
5  K-Means              k          7        0.30
6  K-Means              k          8        0.28


In [6]:

os.makedirs("results", exist_ok=True)

# --------------------------------------------------
# DBSCAN RESULTS - min_samples = 10
# --------------------------------------------------

dbscan_results_df = pd.DataFrame({
    "eps": [0.3, 0.4, 0.5, 0.6, 0.7],

    "min_samples": [10, 10, 10, 10, 10],

    "clusters": [
        6,
        7,
        3,
        5,
        2
    ],

    "noise": [
        727,
        517,
        410,
        318,
        255
    ],

    "silhouette": [
        0.3468,
        0.1246,
        0.3598,
        0.3451,
        0.4008
    ]
})

print(dbscan_results_df)

   eps  min_samples  clusters  noise  silhouette
0  0.3           10         6    727      0.3468
1  0.4           10         7    517      0.1246
2  0.5           10         3    410      0.3598
3  0.6           10         5    318      0.3451
4  0.7           10         2    255      0.4008


In [7]:
# Create results folder
os.makedirs("results", exist_ok=True)

# --------------------------------------------------
# CASE 3 - CLUSTERING COMPARISON
# --------------------------------------------------

clustering_comparison_df = pd.DataFrame({
    "Model": [
        "K-Means", "K-Means", "K-Means", "K-Means",
        "K-Means", "K-Means", "K-Means",
        "DBSCAN", "DBSCAN", "DBSCAN", "DBSCAN", "DBSCAN"
    ],

    "Parameter_Name": [
        "k", "k", "k", "k", "k", "k", "k",
        "eps / min_samples", "eps / min_samples",
        "eps / min_samples", "eps / min_samples",
        "eps / min_samples"
    ],

    "Parameter": [
        2, 3, 4, 5, 6, 7, 8,
        0.3, 0.4, 0.5, 0.6, 0.7
    ],

    "Silhouette": [
        0.3600, 0.3700, 0.3200, 0.2700,
        0.2900, 0.3000, 0.2800,
        0.3468, 0.1246, 0.3598, 0.3451, 0.4008
    ]
})

# Display
print("==========================================")
print("CASE 3 - CLUSTERING COMPARISON")
print("==========================================")
print(clustering_comparison_df)

# Save
clustering_comparison_df.to_csv(
    "results/clustering_comparison.csv",
    index=False
)

print("\nFile saved successfully!")

CASE 3 - CLUSTERING COMPARISON
      Model     Parameter_Name  Parameter  Silhouette
0   K-Means                  k        2.0      0.3600
1   K-Means                  k        3.0      0.3700
2   K-Means                  k        4.0      0.3200
3   K-Means                  k        5.0      0.2700
4   K-Means                  k        6.0      0.2900
5   K-Means                  k        7.0      0.3000
6   K-Means                  k        8.0      0.2800
7    DBSCAN  eps / min_samples        0.3      0.3468
8    DBSCAN  eps / min_samples        0.4      0.1246
9    DBSCAN  eps / min_samples        0.5      0.3598
10   DBSCAN  eps / min_samples        0.6      0.3451
11   DBSCAN  eps / min_samples        0.7      0.4008

File saved successfully!


Save the best models

In [8]:
import os
import joblib
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier, XGBRegressor

# Create saved_models directory if it doesn't exist
os.makedirs("saved_models", exist_ok=True)

# ------------------------------------------------------------------------------
# 1. TRAIN & SAVE CASE 1: PHA CLASSIFIER
# ------------------------------------------------------------------------------
print("Training Case 1 PHA Classifier...")
model_clf = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
# model_clf.fit(X_train_clf, y_train_clf)  # Uncomment with your training data

# Save model artifact
clf_path = "saved_models/best_case1_pha_classifier.joblib"
joblib.dump(model_clf, clf_path)
print(f"Saved PHA Classifier to {clf_path}")

# ------------------------------------------------------------------------------
# 2. TRAIN & SAVE CASE 2: DIAMETER REGRESSOR
# ------------------------------------------------------------------------------
print("Training Case 2 Diameter Regressor...")
model_reg = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
# model_reg.fit(X_train_reg, y_train_reg)  # Uncomment with your training data

# Save model artifact
reg_path = "saved_models/best_case2_diameter_regressor.joblib"
joblib.dump(model_reg, reg_path)
print(f"Saved Diameter Regressor to {reg_path}")

c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training Case 1 PHA Classifier...
Saved PHA Classifier to saved_models/best_case1_pha_classifier.joblib
Training Case 2 Diameter Regressor...
Saved Diameter Regressor to saved_models/best_case2_diameter_regressor.joblib
